In [ ]:
!pip install transformers datasets peft accelerate sacrebleu sentencepiece


In [ ]:
!pip install torchao==0.16.0

In [ ]:
!pip install evaluate sacrebleu rouge_score


In [31]:
import os
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, TaskType, get_peft_model

# =====================================================================
# 1. SETUP & 5-ROW BADAGA (IN TAMIL SCRIPT) DATASET
# =====================================================================
# mT5-base provides robust vocabulary embeddings for Tamil Unicode blocks
MODEL_ID = "google/mt5-base"
SOURCE_KEY = "english"
TARGET_KEY = "badaga"
MAX_LENGTH = 64

# Your absolute 5-row python list of dicts (Badaga represented via Tamil script)
my_custom_data = [
    {"english": "I am here.", "badaga": "நான் இங்கே இருக்கிறேன்."},
    {"english": "you are here.", "badaga": "நீங்கள் இங்கே இருக்கிறீர்கள்"},
    {"english": "we are here.", "badaga": "நாங்கள் இங்கே இருக்கிறோம்"},
    {"english": "I am there", "badaga": "நான் அங்கே இருக்கிறேன்"},
    {"english": "You are there.", "badaga": "நீங்கள் அங்கே இருக்கிறீர்கள்"}
]

# Load list to Dataset.
# NOTE: Because we only have 5 rows, we skip splitting to avoid empty test validation tensors.
full_dataset = Dataset.from_list(my_custom_data)

print("Downloading and preparing Multilingual T5 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)

# =====================================================================
# 2. DATA TOKENIZATION & LOSS PROTECTION
# =====================================================================
def preprocess_function(examples):
    # Formulate prompts that declare the exact target minority language context
    inputs = [f"translate English to Badaga: {src.strip()}" for src in examples[SOURCE_KEY]]
    targets = [tgt.strip() for tgt in examples[TARGET_KEY]]

    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, padding="max_length", truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LENGTH, padding="max_length", truncation=True)

    # Map pad tokens to -100 so the cross-entropy engine prevents blank emissions
    labels_ids = labels["input_ids"]
    cleaned_labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels_ids
    ]

    model_inputs["labels"] = cleaned_labels_ids
    return model_inputs

tokenized_dataset = full_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=[SOURCE_KEY, TARGET_KEY]
)

# =====================================================================
# 3. INITIALIZE MULTILINGUAL BASE MODEL & LORA
# =====================================================================
print("\n--- Loading Pretrained mT5 Architecture ---")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Configure LoRA Parameters for low resource text alignment
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                             # Slightly elevated rank to capture specific phonemes
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "v"]         # Targets attention states
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

# =====================================================================
# 4. TRAINING WITH MODERN ARGUMENTS (OVER-FITTING FOR 5 SAMPLES)
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./lora-badaga-run",
    eval_strategy="epoch",
    learning_rate=1e-3,               # High learning rate to enforce memorisation on tiny data
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=300,              # High epochs so 5 lines can settle inside the adapter weights
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,   # Pointing validation to train set to avoid empty evaluation errors
    processing_class=tokenizer,      # Cleaned keyword argument fix
    data_collator=data_collator,
)

print("\n--- Starting Fine-tuning Pipeline ---")
trainer.train()

# Save final local artifacts
model.save_pretrained("./badaga_lora_adapter")
tokenizer.save_pretrained("./badaga_lora_adapter")
print("Badaga Tamil-script adapters saved successfully.")



Map:   0%|          | 0/5 [00:00<?, ? examples/s]


--- Loading Pretrained mT5 Architecture ---


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 1,769,472 || all params: 584,170,752 || trainable%: 0.3029

--- Starting Fine-tuning Pipeline ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,16.253538
2,19.069200,16.161257
3,19.069200,15.177177
4,17.496031,18.152702
5,18.998572,17.116398
6,18.998572,17.240557
7,18.666702,15.870171
8,18.666702,15.683497
9,18.683449,15.633176
10,20.121263,17.586580


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Badaga Tamil-script adapters saved successfully.


In [33]:
# =====================================================================
# 6. INFERENCE TESTING ON TRAINED SENTENCES
# =====================================================================
print("\n--- Running Inference Testing ---")
unseen_samples = [
    "I am  here.",
    "we are here",
    "we are there"
]

for idx, text in enumerate(unseen_samples, 1):
    infer_prompt = f"translate English to Badaga: {text}"
    inputs = tokenizer(infer_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    translated_text = tokenizer.decode(outputs, skip_special_tokens=True)
    print(f"\n[Inference Sample {idx}]")
    print(f"  Source (EN): {text}")
    print(f"  Result (Badaga/Tamil Script): {translated_text}")



--- Running Inference Testing ---

[Inference Sample 1]
  Source (EN): I am  here.
  Result (Badaga/Tamil Script): ['நான் இங்கே இருக்கிறேன்.']

[Inference Sample 2]
  Source (EN): we are here
  Result (Badaga/Tamil Script): ['நாங்கள் இங்கே இருக்கிறோம்']

[Inference Sample 3]
  Source (EN): we are there
  Result (Badaga/Tamil Script): ['நாங்கள் அங்கே இருக்கிறோம்']


In [ ]:
# =====================================================================
# 5. VALIDATION TESTING WITH EVALUATE ENGINE
# =====================================================================
print("\n--- Running Token Verification Performance Checks ---")
chrf_metric = evaluate.load("chrf")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

predictions = []
references = []

for item in full_dataset:
    src_text = item[SOURCE_KEY]
    ref_text = item[TARGET_KEY]

    prompt = f"translate English to Badaga: {src_text}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=2,
            early_stopping=True
        )

    pred_text = tokenizer.decode(outputs, skip_special_tokens=True)
    predictions.append(pred_text)
    references.append([ref_text])

chrf_results = chrf_metric.compute(predictions=predictions, references=references)
print(f"Validation Overfit ChrF Score: {chrf_results['score']:.2f}")

